In [25]:
import pandas as pd
import numpy as np
diabetes = pd.read_csv('data/diabetes_data.csv')
diabetes.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Gender
0,6,98,58,33,190,34.0,0.430,43,0,Female
1,2,112,75,32,0,35.7,0.148,21,0,Female
2,2,108,64,0,0,30.8,0.158,21,0,Female
3,8,107,80,0,0,24.6,0.856,34,0,Female
4,7,136,90,0,0,29.9,0.210,50,0,Female


Pregnancies — количество беременностей.

Glucose — концентрация глюкозы в плазме через два часа при пероральном тесте на толерантность к глюкозе.

BloodPressure — диастолическое артериальное давление (мм рт. ст.).

SkinThickness — толщина кожной складки трицепса (мм).

Insulin — двухчасовой сывороточный инсулин (ме Ед/мл).

BMI — индекс массы тела (вес в кг делить на рост в метрах в квадрате)

DiabetesPedigreeFunction — функция родословной диабета (чем она выше, тем выше шанс наследственной заболеваемости).

Age — возраст.

Outcome — наличие диабета (0 — нет, 1 — да).

In [26]:
# Задание 8.1
# 1 point possible (graded)
# Начнём с поиска дубликатов в данных. Найдите все повторяющиеся строки в данных и удалите их. Для поиска используйте все признаки в данных. Сколько записей осталось в данных?

print(len(diabetes))
diabetes = diabetes.drop_duplicates()
print(len(diabetes))

778
768


In [27]:
# Задание 8.2
# Далее найдите все неинформативные признаки в данных и избавьтесь от них. В качестве порога информативности возьмите 0.95: удалите все признаки, для которых 95 % значений повторяются или 95 % записей уникальны. В ответ запишите имена признаков, которые вы нашли (без кавычек).
threshold = 0.95

non_informative = []

for col in diabetes.columns:
    n = len(diabetes)
    # доля самого частого значения
    top_share = diabetes[col].value_counts(normalize=True).iloc[0]
    # доля уникальных значений
    unique_share = diabetes[col].nunique() / n

    if top_share >= threshold or unique_share >= threshold:
        non_informative.append(col)

print("Неинформативные признаки:", non_informative)

# Удаляем их
diabetes = diabetes.drop(columns=non_informative)
print("Осталось признаков:", diabetes.shape[1])
print(diabetes.columns.tolist())

Неинформативные признаки: ['Gender']
Осталось признаков: 9
['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']


In [28]:
# Задание 8.3
# Попробуйте найти пропуски в данных с помощью метода isnull().
# Спойлер: ничего не найдёте. А они есть! Просто они скрыты от наших глаз. В таблице пропуски в столбцах Glucose, BloodPressure, SkinThickness, Insulin и BMI обозначены нулём, поэтому традиционные методы поиска пропусков ничего вам не покажут. Давайте это исправим!
# Замените все записи, равные 0, в столбцах Glucose, BloodPressure, SkinThickness, Insulin и BMI на символ пропуска. Его вы можете взять из библиотеки numpy: np.nan.
# Какая доля пропусков содержится в столбце Insulin? Ответ округлите до сотых.

cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
diabetes[cols] = diabetes[cols].replace(0, np.nan)

insulin_share = diabetes['Insulin'].isna().mean()
print(round(insulin_share, 2))

0.49


In [29]:
# Задание 8.4
# Удалите из данных признаки, где число пропусков составляет более 30 %. Сколько признаков осталось в ваших данных (с учетом удаленных неинформативных признаков в задании 8.2)?

# Находим долю пропусков в каждом столбце
missing_ratio = diabetes.isna().mean()

# Оставляем только те, где пропусков не более 30%
cols_to_drop = missing_ratio[missing_ratio > 0.30].index.tolist()
print("Удаляем признаки:", cols_to_drop)

diabetes = diabetes.drop(columns=cols_to_drop)
print("Осталось признаков:", diabetes.shape[1])

Удаляем признаки: ['Insulin']
Осталось признаков: 8


In [31]:
# Задание 8.5
# Удалите из данных только те строки, в которых содержится более двух пропусков одновременно. Чему равно результирующее число записей в таблице?

# Удаляем строки, где количество пропусков строго больше 2

diabetes = diabetes[diabetes.isna().sum(axis=1) <= 2]
print("Результирующее число записей:", len(diabetes))

Результирующее число записей: 761


In [32]:
# Задание 8.6
# В оставшихся записях замените пропуски на медиану. Чему равно среднее значение в столбце SkinThickness? Ответ округлите до десятых.

# 6. Заменяем оставшиеся пропуски на медиану
diabetes = diabetes.fillna(diabetes.median())

# 7. Считаем среднее значение в столбце SkinThickness
mean_skin_thickness = diabetes['SkinThickness'].mean()
print("Среднее значение SkinThickness:", round(mean_skin_thickness, 1))

Среднее значение SkinThickness: 29.1


In [33]:
# Задание 8.7
# Сколько выбросов найдёт классический метод межквартильного размаха в признаке SkinThickness?

# --- Поиск выбросов в SkinThickness ---
Q1 = diabetes['SkinThickness'].quantile(0.25)
Q3 = diabetes['SkinThickness'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Фильтруем значения, выходящие за пределы
outliers = diabetes[(diabetes['SkinThickness'] < lower_bound) | (diabetes['SkinThickness'] > upper_bound)]

print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Границы: [{lower_bound}, {upper_bound}]")
print("Количество выбросов:", len(outliers))

Q1: 25.0, Q3: 32.0, IQR: 7.0
Границы: [14.5, 42.5]
Количество выбросов: 87


In [34]:
# Задание 8.8
# Сколько выбросов найдёт классический метод z-отклонения в признаке SkinThickness?

# --- Метод z-отклонения для SkinThickness ---
mean_st = diabetes['SkinThickness'].mean()
std_st = diabetes['SkinThickness'].std()

# Считаем z-оценки
z_scores = (diabetes['SkinThickness'] - mean_st) / std_st

# Находим количество выбросов (модуль z-оценки > 3)
outliers_count = (abs(z_scores) > 3).sum()

print(f"Среднее: {mean_st:.2f}, Стандартное отклонение: {std_st:.2f}")
print("Количество выбросов по методу z-score:", outliers_count)

Среднее: 29.11, Стандартное отклонение: 8.83
Количество выбросов по методу z-score: 4


In [35]:
# На приведённой гистограмме показано распределение признака DiabetesPedigreeFunction. 
# Такой вид распределения очень похож на логнормальный, и он заставляет задуматься о логарифмировании признака. 
# Найдите сначала число выбросов в признаке DiabetesPedigreeFunction с помощью классического метода межквартильного размаха.
# Затем найдите число выбросов в этом же признаке в логарифмическом масштабе (при логарифмировании единицу прибавлять не нужно!). 
# Какова разница между двумя этими числами (вычтите из первого второе)?

# --- 2. Функция для поиска выбросов методом IQR ---
def count_iqr_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    # Считаем количество значений, выходящих за пределы
    outliers = series[(series < lower_bound) | (series > upper_bound)]
    return len(outliers)

# --- 3. Считаем выбросы в исходном признаке ---
dpf = diabetes['DiabetesPedigreeFunction']
outliers_original = count_iqr_outliers(dpf)

# --- 4. Считаем выбросы в логарифмическом масштабе ---
# Согласно заданию, логарифмируем без прибавления единицы
dpf_log = np.log(dpf)
outliers_log = count_iqr_outliers(dpf_log)

# --- 5. Разница ---
difference = outliers_original - outliers_log

print(f"Выбросов в исходном масштабе: {outliers_original}")
print(f"Выбросов в лог-масштабе: {outliers_log}")
print(f"Разница: {difference}")

Выбросов в исходном масштабе: 29
Выбросов в лог-масштабе: 0
Разница: 29
